# Linear Regression

In [ ]:
import warnings
from scipy.stats import ConstantInputWarning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from analyses.spike_count import aggregate_trial_level, get_binned_spike_trials
import statsmodels.formula.api as smf
import statsmodels.api as sm

%matplotlib inline
%load_ext autoreload
%autoreload 2
warnings.simplefilter("ignore", ConstantInputWarning)

### Get Column Sum and Row Sum from Behavioral Table

In [ ]:
base_dir = '../social_data/zombies_social_data/'
zombies_affiliation_file_name = 'zombies_feature_df_affiliation.xlsx'
zombies_submission_file_name = 'zombies_feature_df_submission.xlsx'
zombies_agonism_file_name = 'zombies_feature_df_agonism.xlsx'

zombies_affiliation_df = pd.read_excel(base_dir + zombies_affiliation_file_name)
zombies_submission_df = pd.read_excel(base_dir + zombies_submission_file_name)
zombies_agonism_df = pd.read_excel(base_dir + zombies_agonism_file_name)

zombies_affiliation_to = zombies_affiliation_df.iloc[:, 1:].to_numpy()
zombies_affiliation_from = zombies_affiliation_to.T

zombies_submission_to = zombies_submission_df.iloc[:, 1:].to_numpy()
zombies_submission_from = zombies_submission_to.T

zombies_agonism_to = zombies_agonism_df.iloc[:, 1:].to_numpy()
zombies_agonism_from = zombies_agonism_to.T

In [ ]:
monkey_names = zombies_affiliation_df.iloc[:, 0].tolist()

In [ ]:
# convert and save
affiliation_to_sum = zombies_affiliation_to.sum(axis=0)
affiliation_to_z = zscore(affiliation_to_sum)
affiliation_to_df = pd.DataFrame({
    'MonkeyName': monkey_names,
    'AffiliationTo': affiliation_to_sum,
    'AffiliationTo_z': affiliation_to_z
})
# affiliation_to_df.to_csv("zombies_affiliation_to.csv", index=False)

### Load the csv sums

In [ ]:
base_dir = '../social_data/zombies_social_data/'
zombies_affiliation_from_file_name = 'zombies_affiliation_from.csv'
zombies_submission_from_file_name = 'zombies_submission_from.csv'
zombies_agonism_from_file_name = 'zombies_agonism_from.csv'
zombies_affiliation_to_file_name = 'zombies_affiliation_to.csv'
zombies_submission_to_file_name = 'zombies_submission_to.csv'
zombies_agonism_to_file_name = 'zombies_agonism_to.csv'
zombies_aff_from = pd.read_csv(base_dir + zombies_affiliation_from_file_name)
zombies_aff_to = pd.read_csv(base_dir + zombies_affiliation_to_file_name)
zombies_sub_from = pd.read_csv(base_dir + zombies_submission_from_file_name)
zombies_sub_to = pd.read_csv(base_dir + zombies_submission_to_file_name)
zombies_agon_from = pd.read_csv(base_dir + zombies_agonism_from_file_name)
zombies_agon_to = pd.read_csv(base_dir + zombies_agonism_to_file_name)

## Trial-Level GLM ( ! Be cautious of collinearity)

In [ ]:
binned_df = get_binned_spike_trials(date='2023-09-26', round_no=1, bin_size=0.05)
trial_df = aggregate_trial_level(binned_df)


In [ ]:
merged_df = pd.merge(trial_df, zombies_aff_to[['MonkeyName', 'AffiliationTo_z']], on='MonkeyName')

In [ ]:
model = smf.ols("SpikeCount ~ AffiliationTo_z", data=merged_df).fit()
print(model.summary())
# plt.scatter(model.fittedvalues, model.resid)
# plt.axhline(0, color='gray', linestyle='--')
# plt.title("OLS residuals")
# plt.xlabel("Fitted values")
# plt.ylabel("Residuals")
# plt.show()

In [ ]:
# trial-level regression
model = smf.glm("SpikeCount ~ AffiliationTo_z", data=merged_df, family=sm.families.Poisson()).fit()
print(model.summary())

## Simple Linear Regression

In [ ]:
binned_df = get_binned_spike_trials(date='2023-09-26', round_no=1, bin_size=0.05)
mean_df = (
    binned_df
    .groupby(['NeuronID', 'MonkeyName'])['SpikeCount']
    .mean()
    .reset_index()
    .rename(columns={'SpikeCount': 'MeanSpikeRate'})
)
merged_df = pd.merge(mean_df, zombies_aff_to[['MonkeyName', 'AffiliationTo_z']], on='MonkeyName')
model = smf.ols("MeanSpikeRate ~ AffiliationTo_z", data=merged_df).fit()
print(model.summary())